# Baseline: glm-4.5 vs the market crowd on the gym's 33-market panel

Two **agent** contestants on the 33 resolved Manifold binary markets in `loom/gym/market_seed_tasks.py` (balanced 17 YES / 16 NO; all admissible for glm-4.5 — each `as_of` ≥ 2024-09, model knowledge cutoff 2024-06-30):

- **no-archive** — Docker sandbox with `network_mode: none`; the agent forecasts from the truncated `/data` dossier + its own knowledge.
- **archive** — same sandbox, but its only route is the date-clamped wayback proxy → the in-cluster pull-through cache → the Internet Archive (it can research "today's" archived internet).

We compare both against the **market crowd** (`prob_at_as_of`, the market's own probability). Each forecast is scored with the gym's proper losses. Every mean carries a 95% percentile **cluster-bootstrap CI clustered by `as_of`** (markets sharing an era are correlated), and we report **paired per-task deltas** — which difference out shared task difficulty and are far more powerful than comparing two absolute means at n=33.

Reproduce the runs (glm-4.5 via the cluster LiteLLM) and point this notebook at their Inspect log dirs:

```bash
agent_eval_bin --model-id glm-4.5 --no-archive --task-filter manifold- --log-dir <noarch>
agent_eval_bin --model-id glm-4.5            --task-filter manifold- \
    --wayback-upstream https://wayback-cache.allegedly.works --log-dir <arch>
```

In [ ]:
from loom.gym.analyze_baselines import aggregate, as_of_clusters, crowd_baseline, paired_delta, read_run

# Inspect `.eval` log dirs (local, or s3://loom-gym/agent-runs/...).
NOARCH_DIR = "/tmp/baseline2/noarch"
ARCH_DIR = "/tmp/baseline2/arch"

runs = {"no-archive": read_run(NOARCH_DIR), "archive": read_run(ARCH_DIR), "market crowd": crowd_baseline()}
clusters = as_of_clusters()

## Absolute means (95% cluster-bootstrap CI)

In [ ]:
print(f"{'contestant':<14}{'n':>8}{'log loss (95% CI)':>26}{'Brier (95% CI)':>24}")
for label, res in runs.items():
    ll, llci, n_sub, n_tot = aggregate(res, clusters, "log_loss")
    br, brci, _, _ = aggregate(res, clusters, "brier")
    print(f"{label:<14}{f'{n_sub}/{n_tot}':>8}{ll:>9.3f} [{llci[0]:.3f}, {llci[1]:.3f}]   {br:>6.3f} [{brci[0]:.3f}, {brci[1]:.3f}]")

## Paired per-task deltas (b − a; negative = b better)

These cancel the shared task/era difficulty, so the same clusters resolve a much smaller effect with a tighter CI.

In [ ]:
for a, b in [("no-archive", "archive"), ("market crowd", "no-archive"), ("market crowd", "archive")]:
    for metric, name in (("log_loss", "log loss"), ("brier", "Brier")):
        d, ci, n = paired_delta(runs[a], runs[b], clusters, metric)
        print(f"{b} - {a:<13} {name:<9} {d:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}]  (n={n})")

## Plots

In [ ]:
import matplotlib.pyplot as plt

labels = list(runs)
stats = {k: aggregate(runs[k], clusters, "log_loss") for k in labels}
means = [stats[k][0] for k in labels]
lo = [stats[k][0] - stats[k][1][0] for k in labels]
hi = [stats[k][1][1] - stats[k][0] for k in labels]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, means, yerr=[lo, hi], capsize=6, color=["#4c72b0", "#dd8452", "#999999"])
ax.set_ylabel("mean log loss (lower is better)")
ax.set_title("glm-4.5 on 33 resolved Manifold markets\n95% cluster-bootstrap CI (clustered by as_of)")
fig.tight_layout()

## Findings (run of 2026-06-11, glm-4.5)

| contestant | n | log loss (95% CI) | Brier |
| --- | --- | --- | --- |
| no-archive | 32/33 | 0.903 [0.626, 1.255] | 0.311 |
| archive | **20/33** | 1.059 [0.580, 1.595] | 0.339 |
| market crowd | 33/33 | 0.796 [0.596, 0.993] | 0.278 |

Paired deltas: `archive − no-archive` = **+0.152** [−0.213, +0.479] (n=19); `no-archive − crowd` = +0.116 [−0.130, +0.398] (n=32); `archive − crowd` = +0.122 [−0.344, +0.585] (n=20).

**What's solid.** glm-4.5 is **statistically indistinguishable from the market crowd** — both agent arms' paired deltas vs. the crowd straddle 0. A capable model with only the dossier + its own knowledge matches the Manifold crowd on these resolved markets.

**What is NOT yet trustworthy: whether the archive helps.** The `archive − no-archive` delta (+0.152, straddles 0) is confounded this run because **archive access was degraded by Internet-Archive HTTP 502s** (~40 failed CDX lookups). Consequences:

1. **Archive produced only 20/33 submissions** (13 non-submissions / NaN) vs. 1/33 for no-archive — the agent burned its fixed turn budget retrying failing fetches and never submitted. So the archive arm is low-n with selection bias toward tasks that happened to finish.
2. This is therefore *flaky archive* vs. no-archive, not *archive* vs. no-archive.

**Do not conclude the archive is useless.** Conclude that a *degraded* archive plus a fixed turn budget hurts. A trustworthy `archive − no-archive` needs the cache reliability fixes first (retry IA 502s on the cache hop; capture upstream error bodies for diagnosis), and likely a higher message limit and/or a "submit a best guess before you run out of turns" nudge so a flaky archive can't manufacture non-submissions.